# Prompt Engineering Portfolio — Week 1
### GenAI Roadmap — Week 1 Mini-Project

Demonstrates **12 prompt engineering techniques** applied across four task types — text classification, summarization, code generation, and data extraction — with outputs compared across **Groq**, **api.airforce**, and a **local Ollama model**.

This notebook is intentionally thin: prompts live in [`prompts/`](./prompts), reusable logic lives in [`src/`](./src), and each run's results are saved to [`outputs/`](./outputs). See [README.md](./README.md) for the full layout.

| # | Technique | Task type |
|---|-----------|-----------|
| 1 | Zero-Shot Prompting | Text classification |
| 2 | Few-Shot Prompting | Text classification |
| 3 | Chain-of-Thought (CoT) | Data extraction / reasoning |
| 4 | System Message & Role Assignment | Summarization |
| 5 | Structured Output (JSON) | Data extraction |
| 6 | ReAct Pattern (reason + act with a tool) | Reasoning / tool use |
| 7 | Prompt Chaining | Summarization |
| 8 | Code Generation Prompting | Code generation |
| 9 | Reflection / Iterative Refinement | Code generation |
| 10 | Role-Playing / Persona Prompting | Text generation |
| 11 | Self-Consistency (sample & vote) | Reasoning |
| 12 | Cross-Model Comparison | All tasks |

## Setup

1. Copy `.env.example` to `.env` in this folder and fill in your keys:
   - `GROQ_API_KEY`
   - `AIRFORCE_API_KEY`
2. Install dependencies: `pip install -r requirements.txt`
3. For the local model, install [Ollama](https://ollama.ai), run `ollama serve`, then pull a model:
   `ollama pull llama3.1`
4. Run this notebook with its working directory set to `week-1/` (the default when opening it directly from this folder in Jupyter or VS Code) so `prompts` and `src` resolve as importable packages.

In [ ]:
import json

import pandas as pd

from src.model_clients import compare_models, show
from src.persistence import save_result
from src.techniques import run_prompt_chain, run_react, run_reflection, run_self_consistency

## 1. Zero-Shot Prompting
*Task: Text classification.* Direct instructions, no examples — the baseline for every task.

In [ ]:
from prompts.zero_shot import PROMPT

results = compare_models(PROMPT)
save_result("01_zero_shot", {"prompt": PROMPT, "responses": results})
show(results)

## 2. Few-Shot Prompting
*Task: Text classification.* A handful of labeled examples steers the format and category boundaries.

In [ ]:
from prompts.few_shot import PROMPT

results = compare_models(PROMPT)
save_result("02_few_shot", {"prompt": PROMPT, "responses": results})
show(results)

## 3. Chain-of-Thought (CoT) Prompting
*Task: Data extraction / arithmetic reasoning.* Explicit "think step by step" reasoning before the final answer.

In [ ]:
from prompts.chain_of_thought import PROMPT

results = compare_models(PROMPT)
save_result("03_chain_of_thought", {"prompt": PROMPT, "responses": results})
show(results)

## 4. System Message & Role Assignment
*Task: Summarization.* A system prompt sets persona and constraints that shape every response.

In [ ]:
from prompts.system_role import PROMPT, SYSTEM

results = compare_models(PROMPT, system=SYSTEM)
save_result("04_system_role", {"prompt": PROMPT, "system": SYSTEM, "responses": results})
show(results)

## 5. Structured Output (JSON)
*Task: Data extraction.* Requesting a strict schema for reliable, parseable output.

In [ ]:
from prompts.structured_output import PROMPT

results = compare_models(PROMPT)
parsed = {}
for name, text in results.items():
    print(f"--- {name} ---")
    try:
        parsed[name] = json.loads(text)
        print(json.dumps(parsed[name], indent=2))
    except json.JSONDecodeError:
        print("(not valid JSON, raw output below)")
        print(text)
    print()

save_result("05_structured_output", {"prompt": PROMPT, "responses": results, "parsed": parsed})

## 6. ReAct Pattern (Reason + Act)
*Task: Reasoning with tool use.* The model alternates Thought → Action → Observation, calling a `calculator` tool that `src.techniques.run_react` executes on its behalf (via `src.safe_eval`, not `eval`) and feeds back as an observation.

In [ ]:
from prompts.react import INSTRUCTIONS, QUESTION

transcript = run_react(QUESTION, INSTRUCTIONS)
save_result("06_react", {"question": QUESTION, "transcript": transcript})
print(transcript)

## 7. Prompt Chaining
*Task: Summarization → transformation.* Break the task into sequential prompts, each building on the previous output.

In [ ]:
from prompts.prompt_chaining import SOURCE_TEXT, summary_prompt, tweet_prompt

result = run_prompt_chain(SOURCE_TEXT, summary_prompt, tweet_prompt)
save_result("07_prompt_chaining", {"source_text": SOURCE_TEXT, **result})
print("Summary:", result["summary"])
print("\nTweet:", result["tweet"])

## 8. Code Generation Prompting
*Task: Code generation.* Ask for a function with an explicit contract: signature, docstring, and test cases.

In [ ]:
from prompts.code_generation import PROMPT

results = compare_models(PROMPT)
save_result("08_code_generation", {"prompt": PROMPT, "responses": results})
show(results)

## 9. Reflection / Iterative Refinement
*Task: Code generation.* The model critiques its own first draft (reusing the Technique 8 prompt), then produces a corrected version — catching edge cases a single pass tends to miss.

In [ ]:
from prompts.reflection import CODE_GEN_PROMPT, critique_prompt

result = run_reflection(CODE_GEN_PROMPT, critique_prompt)
save_result("09_reflection", result)
print("--- First draft ---\n", result["first_pass"])
print("\n--- Critique + revision ---\n", result["critique"])

## 10. Role-Playing / Persona Prompting
*Task: Text generation.* Assigning a persona in the prompt itself (rather than the system message) reshapes tone and style.

In [ ]:
from prompts.persona import PROMPT

results = compare_models(PROMPT)
save_result("10_persona", {"prompt": PROMPT, "responses": results})
show(results)

## 11. Self-Consistency
*Task: Reasoning.* Sample the same reasoning prompt multiple times and take the majority answer — more robust than trusting a single completion.

In [ ]:
from prompts.self_consistency import PROMPT, QUESTION

result = run_self_consistency(PROMPT)
save_result("11_self_consistency", {"question": QUESTION, **result})
for i, s in enumerate(result["samples"], 1):
    print(f"Sample {i}:\n{s}\n")
print("Answer distribution:", result["distribution"])
print("Majority answer:", result["majority"])

## 12. Cross-Model Comparison
*Task: All task types.* The same prompt run through Groq, api.airforce, and a local Ollama model side by side, so differences in style, accuracy, and verbosity are easy to compare directly.

In [ ]:
from prompts.cross_model import PROMPT

results = compare_models(PROMPT)
save_result("12_cross_model", {"prompt": PROMPT, "responses": results})
df = pd.DataFrame(list(results.items()), columns=["Model", "Response"])
df

## Takeaways

_Fill this in after running the notebook with real API keys:_

- Which technique improved output quality the most for each task type?
- Where did Groq, api.airforce, and the local Ollama model disagree or differ in quality?
- Which technique was most worth the added prompt complexity, and which wasn't?